In [1]:
import os
import glob
import pandas as pd
from pathlib import Path

In [2]:
BASE_DIR = "cachaca"  # ajuste se necessário
REPORTS_DIR = os.path.join(BASE_DIR, "reports_out")
Path(REPORTS_DIR).mkdir(parents=True, exist_ok=True)


# util simples para listar subpastas imediatas
def list_subdirs(path):
    return sorted([p for p in os.listdir(path) if (Path(path) / p).is_dir()])

In [3]:
INS_DIR = os.path.join(BASE_DIR, "insights_out")
splits_insights = list_subdirs(INS_DIR)

In [4]:
def load_insights(base_dir):
    out = {}
    for split in list_subdirs(base_dir):
        sdir = os.path.join(base_dir, split)
        try:
            q = pd.read_csv(os.path.join(sdir, "length_quantiles.csv"))
            q.insert(0, "split", split)
        except FileNotFoundError:
            q = None
        try:
            oov = pd.read_csv(os.path.join(sdir, "oov_rates.csv"))
            oov.insert(0, "split", split)
        except FileNotFoundError:
            oov = None
        try:
            rare = pd.read_csv(os.path.join(sdir, "rare_labels.csv"))
            rare.insert(0, "split", split)
        except FileNotFoundError:
            rare = None
        try:
            tok_te = pd.read_csv(os.path.join(sdir, "tokens_test.csv"))
            tok_te.insert(0, "split", split)
        except FileNotFoundError:
            tok_te = None
        try:
            tok_tr = pd.read_csv(os.path.join(sdir, "tokens_train.csv"))
            tok_tr.insert(0, "split", split)
        except FileNotFoundError:
            tok_tr = None

        out[split] = {
            "quantiles": q,
            "oov": oov,
            "rare": rare,
            "tokens_test": tok_te,
            "tokens_train": tok_tr,
        }
    return out

In [5]:
ins = load_insights(INS_DIR)

In [6]:
# Tabelas consolidadas simples:
ins_quantiles = pd.concat(
    [ins[s]["quantiles"] for s in ins if ins[s]["quantiles"] is not None],
    ignore_index=True,
)
ins_oov = pd.concat(
    [ins[s]["oov"] for s in ins if ins[s]["oov"] is not None], ignore_index=True
)
ins_rare = pd.concat(
    [ins[s]["rare"] for s in ins if ins[s]["rare"] is not None], ignore_index=True
)
ins_tok_te = pd.concat(
    [ins[s]["tokens_test"] for s in ins if ins[s]["tokens_test"] is not None],
    ignore_index=True,
)
ins_tok_tr = pd.concat(
    [ins[s]["tokens_train"] for s in ins if ins[s]["tokens_train"] is not None],
    ignore_index=True,
)

# OOV em formato largo por métrica
ins_oov_wide = ins_oov.pivot(
    index="split", columns="metric", values="value"
).reset_index()

In [7]:
display(ins_oov_wide)

metric,split,oov_rate_test_vs_train,oov_rate_val_vs_train
0,adversarial,0.048592,0.026988
1,heur_len,0.026766,0.030275
2,heur_rare,0.122513,0.006125
3,loc,0.134162,0.126162
4,reverse,0.094469,0.105275
5,semantic,0.180261,0.098243
6,standard,0.026386,0.024436


In [8]:
CD_DIR = os.path.join(BASE_DIR, "class_dist_out")


def load_class_dist(base_dir):
    rows_counts, rows_props, rows_long = [], [], []
    label_vecs_all = []
    for split in list_subdirs(base_dir):
        sdir = os.path.join(base_dir, split)

        # long
        if Path(os.path.join(sdir, "class_distribution_long.csv")).exists():
            df_long = pd.read_csv(os.path.join(sdir, "class_distribution_long.csv"))
            df_long.insert(0, "split_name", split)
            rows_long.append(df_long)

        # counts e props
        if Path(os.path.join(sdir, "counts_pivot.csv")).exists():
            cnt = pd.read_csv(os.path.join(sdir, "counts_pivot.csv"))
            cnt.insert(0, "split_name", split)
            rows_counts.append(cnt)

        if Path(os.path.join(sdir, "props_pivot.csv")).exists():
            pr = pd.read_csv(os.path.join(sdir, "props_pivot.csv"))
            pr.insert(0, "split_name", split)
            rows_props.append(pr)

        # label_vecs.csv (linha por part)
        if Path(os.path.join(sdir, "label_vecs.csv")).exists():
            lv = pd.read_csv(os.path.join(sdir, "label_vecs.csv"))
            lv.insert(0, "split_name", split)
            label_vecs_all.append(lv)

    long_df = pd.concat(rows_long, ignore_index=True) if rows_long else pd.DataFrame()
    counts = (
        pd.concat(rows_counts, ignore_index=True) if rows_counts else pd.DataFrame()
    )
    props = pd.concat(rows_props, ignore_index=True) if rows_props else pd.DataFrame()
    lvecs = (
        pd.concat(label_vecs_all, ignore_index=True)
        if label_vecs_all
        else pd.DataFrame()
    )
    return long_df, counts, props, lvecs


cd_long, cd_counts, cd_props, cd_lv = load_class_dist(CD_DIR)

In [9]:
merged_wide = cd_counts.merge(
    cd_props,
    on=["split_name", "label"],
    suffixes=("_count", "_prop"),
)
merged_wide.to_csv(
    os.path.join(REPORTS_DIR, "class_counts_props_merged_wide.csv"), index=False
)

In [10]:
props_long = cd_props.melt(
    id_vars=["split_name", "label"],
    var_name="part",
    value_name="prop",
)
props_long_sorted = props_long.sort_values(
    ["split_name", "part", "prop"], ascending=[True, True, False]
)

# counts longo para anexar contagem ao top-5
counts_long = cd_counts.melt(
    id_vars=["split_name", "label"],
    var_name="part",
    value_name="count",
)

In [11]:
top5 = (
    props_long_sorted.groupby(["split_name", "part"], group_keys=False)
    .head(5)
    .merge(counts_long, on=["split_name", "label", "part"], how="left")
)

In [12]:
print("== Merged (wide) ==")
display(merged_wide.head())

== Merged (wide) ==


,split_name,label,test_count,train_count,val_count,test_prop,train_prop,val_prop
0,adversarial,B-CARACTERISTICA_SENSORIAL_AROMA,238,624,73,0.006530,0.004835,0.004165
1,adversarial,B-CARACTERISTICA_SENSORIAL_CONSISTÊNCIA,71,179,28,0.001948,0.001387,0.001598
2,adversarial,B-CARACTERISTICA_SENSORIAL_COR,123,383,56,0.003375,0.002968,0.003195
3,adversarial,B-CARACTERISTICA_SENSORIAL_SABOR,190,620,96,0.005213,0.004804,0.005478
4,adversarial,B-CLASSIFICACAO_BEBIDA,250,944,131,0.006859,0.007315,0.007475


In [13]:
print("\n== Props (long, sorted) ==")
display(props_long_sorted.head(12))


== Props (long, sorted) ==


,split_name,label,part,prop
34,adversarial,O,test,0.777232
8,adversarial,B-NOME_LOCAL,test,0.024502
7,adversarial,B-NOME_BEBIDA,test,0.018082
24,adversarial,I-NOME_BEBIDA,test,0.016600
15,adversarial,B-TIPO_MADEIRA,test,0.014624
28,adversarial,I-PRECO,test,0.014021
16,adversarial,B-VOLUME,test,0.012567
25,adversarial,I-NOME_LOCAL,test,0.010893
13,adversarial,B-TEMPO,test,0.008067
26,adversarial,I-NOME_ORGANIZACAO,test,0.007683


In [14]:
print("\n== Top-5 por split e partição ==")
display(top5)


== Top-5 por split e partição ==


,split_name,label,part,prop,count
0,adversarial,O,test,0.777232,28327
1,adversarial,B-NOME_LOCAL,test,0.024502,893
2,adversarial,B-NOME_BEBIDA,test,0.018082,659
3,adversarial,I-NOME_BEBIDA,test,0.016600,605
4,adversarial,B-TIPO_MADEIRA,test,0.014624,533
...,...,...,...,...,...
100,standard,O,val,0.791929,14227
101,standard,B-NOME_LOCAL,val,0.023880,429
102,standard,B-NOME_BEBIDA,val,0.016755,301
103,standard,I-PRECO,val,0.014862,267


In [28]:
top5['part'].value_counts()

part
test     35
train    35
val      35
Name: count, dtype: int64

In [29]:
parts = ["train", "val", "test"]

for split in sorted(top5["split_name"].unique()):
    print(f"\n=== {split} ===")
    for part in parts:
        sub = (
            top5[(top5["split_name"] == split) & (top5["part"] == part)]
            .sort_values("prop", ascending=False)
            .head(5)
            .loc[:, ["label", "prop", "count"]]
            .reset_index(drop=True)
        )
        print(f"\n[{part}] top-5")
        try:
            display(sub)  # funciona no Jupyter
        except NameError:
            print(sub.to_string(index=False))  # fallback se display não existir


=== adversarial ===

[train] top-5


,label,prop,count
0,O,0.788744,101785
1,B-NOME_LOCAL,0.022480,2901
2,B-NOME_BEBIDA,0.017211,2221
3,I-NOME_BEBIDA,0.014483,1869
4,B-VOLUME,0.013979,1804



[val] top-5


,label,prop,count
0,O,0.781296,13693
1,B-NOME_LOCAL,0.024991,438
2,B-NOME_BEBIDA,0.016604,291
3,B-VOLUME,0.015406,270
4,I-PRECO,0.014265,250



[test] top-5


...

,label,prop,count
0,O,0.785290,114705
1,B-NOME_LOCAL,0.022620,3304
2,B-NOME_BEBIDA,0.017382,2539
3,I-NOME_BEBIDA,0.014993,2190
4,I-PRECO,0.014062,2054



[val] top-5


,label,prop,count
0,O,0.791929,14227
1,B-NOME_LOCAL,0.023880,429
2,B-NOME_BEBIDA,0.016755,301
3,I-PRECO,0.014862,267
4,B-TIPO_MADEIRA,0.013916,250



[test] top-5


,label,prop,count
0,O,0.783325,14873
1,B-NOME_LOCAL,0.026281,499
2,B-NOME_BEBIDA,0.017433,331
3,I-NOME_BEBIDA,0.015853,301
4,B-VOLUME,0.014852,282


In [15]:
CWI_DIR = os.path.join(BASE_DIR, "cosine_out")

# tenta usar o resumo pronto; se não existir, empilha de cada split
summary_path = os.path.join(CWI_DIR, "cosine_summary_all_splits.csv")
if Path(summary_path).exists():
    cos_within_all = pd.read_csv(summary_path)
else:
    rows = []
    for split in list_subdirs(CWI_DIR):
        f = os.path.join(CWI_DIR, split, "cosine_all.csv")
        if Path(f).exists():
            df = pd.read_csv(f)
            df.insert(0, "split", split)
            rows.append(df)
    cos_within_all = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

# Tabelas simples:
# - matriz (a,b) por espaço+split_set (labels/words e pares)
if not cos_within_all.empty:
    cos_within_pivot = (
        cos_within_all.assign(pair=lambda d: d["a"] + "_" + d["b"])
        .pivot_table(
            index=["split", "space"],
            columns="pair",
            values="cosine_distance",
            aggfunc="first",
        )
        .reset_index()
    )
else:
    cos_within_pivot = pd.DataFrame()

In [17]:
cos_within_all.query("space == 'words'")[['split', 'a', 'b', 'cosine_distance']]

,split,a,b,cosine_distance
3,standard,train,val,0.026027
4,standard,train,test,0.023644
5,standard,val,test,0.041253
9,heur_len,train,val,0.027243
10,heur_len,train,test,0.014990
11,heur_len,val,test,0.033095
15,heur_rare,train,val,0.025512
16,heur_rare,train,test,0.257742
17,heur_rare,val,test,0.273369
21,adversarial,train,val,0.024152


In [18]:
cos_within_all.query("space == 'labels'")[["split", "a", "b", "cosine_distance"]]

,split,a,b,cosine_distance
0,standard,train,val,0.000013
1,standard,train,test,0.000034
2,standard,val,test,0.000051
6,heur_len,train,val,0.000035
7,heur_len,train,test,0.000005
8,heur_len,val,test,0.000032
12,heur_rare,train,val,0.000018
13,heur_rare,train,test,0.000504
14,heur_rare,val,test,0.000590
18,adversarial,train,val,0.000019


In [19]:
display(cos_within_pivot)

pair,split,space,train_test,train_val,val_test
0,adversarial,labels,0.000026,0.000019,0.000037
1,adversarial,words,0.056733,0.024152,0.073582
2,heur_len,labels,0.000005,0.000035,0.000032
3,heur_len,words,0.014990,0.027243,0.033095
4,heur_rare,labels,0.000504,0.000018,0.000590
5,heur_rare,words,0.257742,0.025512,0.273369
6,loc,labels,0.001361,0.001172,0.000067
7,loc,words,0.393926,0.343770,0.062198
8,reverse,labels,0.001480,0.009178,0.006413
9,reverse,words,0.502707,0.836843,0.312683


In [20]:
cos_within_pivot.query("space == 'labels'")[
    ["split", "train_test", "train_val", "val_test"]
]

pair,split,train_test,train_val,val_test
0,adversarial,0.000026,0.000019,0.000037
2,heur_len,0.000005,0.000035,0.000032
4,heur_rare,0.000504,0.000018,0.000590
6,loc,0.001361,0.001172,0.000067
8,reverse,0.001480,0.009178,0.006413
10,semantic,0.050908,0.007090,0.030843
12,standard,0.000034,0.000013,0.000051


In [21]:
cos_within_pivot.query("space == 'words'")[
    ["split", "train_test", "train_val", "val_test"]
]

pair,split,train_test,train_val,val_test
1,adversarial,0.056733,0.024152,0.073582
3,heur_len,0.014990,0.027243,0.033095
5,heur_rare,0.257742,0.025512,0.273369
7,loc,0.393926,0.343770,0.062198
9,reverse,0.502707,0.836843,0.312683
11,semantic,0.891905,0.840690,0.486384
13,standard,0.023644,0.026027,0.041253


In [22]:
CB_DIR = os.path.join(BASE_DIR, "cosine_between_out")


def safe_read_csv(path):
    return pd.read_csv(path) if Path(path).exists() else None


cos_bw_train_words = safe_read_csv(os.path.join(CB_DIR, "cos_train_words.csv"))
cos_bw_test_words = safe_read_csv(os.path.join(CB_DIR, "cos_test_words.csv"))
cos_bw_train_labels = safe_read_csv(os.path.join(CB_DIR, "cos_train_labels.csv"))
cos_bw_test_labels = safe_read_csv(os.path.join(CB_DIR, "cos_test_labels.csv"))

In [23]:
cos_bw_train_words

,Unnamed: 0,standard,heur_len,heur_rare,adversarial,loc,semantic,reverse
0,standard,0.000000,0.000581,0.004535,0.004336,0.022794,0.152848,0.049394
1,heur_len,0.000581,0.000000,0.004806,0.004792,0.022430,0.155830,0.048831
2,heur_rare,0.004535,0.004806,0.000000,0.006362,0.011092,0.185070,0.039675
3,adversarial,0.004336,0.004792,0.006362,0.000000,0.023650,0.162707,0.046689
4,loc,0.022794,0.022430,0.011092,0.023650,0.000000,0.253536,0.045687
5,semantic,0.152848,0.155830,0.185070,0.162707,0.253536,0.000000,0.225683
6,reverse,0.049394,0.048831,0.039675,0.046689,0.045687,0.225683,0.000000


In [24]:
cos_bw_test_words

,Unnamed: 0,standard,heur_len,heur_rare,adversarial,loc,semantic,reverse
0,standard,0.000000,0.022591,0.263536,0.070826,0.280190,0.388808,0.313869
1,heur_len,0.022591,0.000000,0.242735,0.052868,0.262759,0.397879,0.285215
2,heur_rare,0.263536,0.242735,0.000000,0.248594,0.058559,0.802478,0.175569
3,adversarial,0.070826,0.052868,0.248594,0.000000,0.262370,0.462096,0.269413
4,loc,0.280190,0.262759,0.058559,0.262370,0.000000,0.862241,0.192907
5,semantic,0.388808,0.397879,0.802478,0.462096,0.862241,0.000000,0.749688
6,reverse,0.313869,0.285215,0.175569,0.269413,0.192907,0.749688,0.000000


In [25]:
cos_bw_train_labels

,Unnamed: 0,standard,heur_len,heur_rare,adversarial,loc,semantic,reverse
0,standard,0.000000,0.000158,0.000107,0.000072,0.000377,0.000386,0.000377
1,heur_len,0.000158,0.000000,0.000149,0.000121,0.000368,0.000231,0.000542
2,heur_rare,0.000107,0.000149,0.000000,0.000030,0.000404,0.000474,0.000455
3,adversarial,0.000072,0.000121,0.000030,0.000000,0.000522,0.000344,0.000417
4,loc,0.000377,0.000368,0.000404,0.000522,0.000000,0.000999,0.000880
5,semantic,0.000386,0.000231,0.000474,0.000344,0.000999,0.000000,0.000504
6,reverse,0.000377,0.000542,0.000455,0.000417,0.000880,0.000504,0.000000


In [26]:
cos_bw_test_labels

,Unnamed: 0,standard,heur_len,heur_rare,adversarial,loc,semantic,reverse
0,standard,0.000000,0.000206,0.000417,0.000129,0.000440,0.045810,0.000643
1,heur_len,0.000206,0.000000,0.000391,0.000128,0.000488,0.044489,0.000747
2,heur_rare,0.000417,0.000391,0.000000,0.000357,0.000109,0.049585,0.000509
3,adversarial,0.000129,0.000128,0.000357,0.000000,0.000529,0.044777,0.000570
4,loc,0.000440,0.000488,0.000109,0.000529,0.000000,0.051890,0.000631
5,semantic,0.045810,0.044489,0.049585,0.044777,0.051890,0.000000,0.046713
6,reverse,0.000643,0.000747,0.000509,0.000570,0.000631,0.046713,0.000000
